# Mesh generation - Gridgen quadtree

Build a **quadtree** unstructured (DISV) grid with the Gridgen program: start from a coarse base grid and recursively quarter cells near the area of interest and along the river network.

Part of the **mesh-generation** series, adapted from the FloPy watershed geoprocessing example (Hughes and others, 2023, *FloPy Workflows for Creating Structured and Unstructured MODFLOW Models*, Groundwater, https://doi.org/10.1111/gwat.13327). Each notebook builds one family of grids over the same synthetic watershed, samples a fine topographic raster onto the grid, and intersects the river network with the grid cells.

- **Rectilinear (DIS + LGR)** (`mesh-generation-rectilinear`) - structured grids: constant and variable spacing, plus local grid refinement
- **Gridgen quadtree** (`mesh-generation-gridgen`) - a quadtree unstructured (DISV) grid built with Gridgen *(this notebook)*
- **Triangle & Voronoi** (`mesh-generation-triangle-voronoi`) - unstructured (DISV) grids built with Triangle and Voronoi

## Imports and setup

Import FloPy and the shared watershed setup from `mesh_helpers`: the problem extent and contour levels, the fine topographic raster, the boundary/river geometry, and the `resample_topo` / `river_intersection` / `draw_boundary_river` helpers that every mesh-generation notebook uses.

In [ ]:
%matplotlib inline
import pathlib as pl

import flopy
import flopy.plot.styles as styles
import matplotlib.pyplot as plt
from mesh_helpers import (
    Lx,
    Ly,
    boundary_polygon,
    draw_boundary_river,
    levels,
    resample_topo,
    river_intersection,
    sgs,
    vmax,
    vmin,
)
from notebook_helpers import set_idomain

## Build the quadtree grid

Start from a coarse (5 km) base grid, then add Gridgen refinement features: a polygon around the area of interest and the river lines. Gridgen writes to a workspace under `models/`.

In [ ]:
from flopy.utils.gridgen import Gridgen

gridgen_ws = pl.Path("models/mesh-generation-gridgen")

# area of interest to refine (the LGR child region from the rectilinear notebook)
refine_poly = [
    [
        (65000.0, 40000.0),
        (65000.0, 60000.0),
        (90000.0, 60000.0),
        (90000.0, 40000.0),
        (65000.0, 40000.0),
    ]
]

# a coarse base grid for Gridgen to refine
sim = flopy.mf6.MFSimulation()
gwf = flopy.mf6.ModflowGwf(sim)
dx = dy = 5000.0
flopy.mf6.ModflowGwfdis(gwf, nrow=int(Ly / dy), ncol=int(Lx / dx), delr=dy, delc=dx)

gridgen_ws.mkdir(parents=True, exist_ok=True)
g = Gridgen(gwf.modelgrid, model_ws=gridgen_ws)
g.add_refinement_features([refine_poly], "polygon", 2, range(1))
g.add_refinement_features(sgs, "line", 2, range(1))
g.build(verbose=False)

quadtree_grid = flopy.discretization.VertexGrid(**g.get_gridprops_vertexgrid())
set_idomain(quadtree_grid, boundary_polygon)

top_qg = resample_topo(quadtree_grid)
intersection_qg = river_intersection(quadtree_grid)

In [ ]:
with styles.USGSMap():
    fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    ax.set_aspect("equal")
    pmv = flopy.plot.PlotMapView(modelgrid=quadtree_grid, ax=ax)
    pmv.plot_array(top_qg, ec="0.75", vmin=vmin, vmax=vmax)
    pmv.plot_array(intersection_qg, masked_values=[0], alpha=0.2, cmap="Reds_r")
    pmv.contour_array(top_qg, levels=levels, linewidths=0.3, colors="white")
    pmv.plot_inactive(zorder=100)
    draw_boundary_river(ax)
    ax.set_title("Gridgen quadtree grid")

**Recap.** Gridgen turns a coarse base grid into a quadtree DISV grid, quartering cells near the area of interest and the river while leaving the rest of the domain coarse. The Triangle and Voronoi notebook builds fully unstructured grids over the same watershed.